In [1]:
import pulp
import pandas as pd

In [2]:
df=pd.read_csv(
"optimization_dataset.csv"
)

df.head()

,Delivery Status,Shipping Mode,delivery_time,profit,inventory_demand,Order Item Quantity
0,Advance shipping,Standard Class,3,91.250000,327.75,1
1,Late delivery,Standard Class,5,-249.089996,327.75,1
2,Shipping on time,Standard Class,4,-247.779999,327.75,1
3,Advance shipping,Standard Class,3,22.860001,327.75,1
4,Advance shipping,Standard Class,2,134.210007,327.75,1


In [3]:
df.shape

(180519, 6)

## Decision Variable

In [4]:
model=pulp.LpProblem(
"Supply_Chain_MILP_Optimization",
pulp.LpMinimize
)

## Create variables:

In [5]:
# 2. Define number of records
n = len(df)


# 3. Define decision variables
x = pulp.LpVariable.dicts(
    "Order",
    range(n),
    cat="Binary"
)


q = pulp.LpVariable.dicts(
    "Inventory",
    range(n),
    lowBound=0,
    cat="Continuous"
)

## Defining Objectives

In [6]:
df["inventory_efficiency"] = (
    df["Order Item Quantity"] /
    df["inventory_demand"]
)

In [7]:
# Objective 1
# Minimize delivery time

delivery = pulp.lpSum(
    df.iloc[i]["delivery_time"] * x[i]
    for i in range(n)
)


# Objective 2
# Maximize profit

profit = pulp.lpSum(
    df.iloc[i]["profit"] * x[i]
    for i in range(n)
)


# Objective 3
# Inventory efficiency

inventory = pulp.lpSum(
    df.iloc[i]["inventory_efficiency"] * q[i]
    for i in range(n)
)

## Weighted Multi Objective Function

In [8]:
# Weighted multi objective

model += (
    0.4*delivery
    -
    0.4*profit
    +
    0.2*inventory
)

## Constraints

In [9]:
# Constraint:
# Select limited orders

model += (
    pulp.lpSum(
        x[i]
        for i in range(n)
    )
    <= 50
)


# Inventory constraint

inventory_capacity = 10000

model += (
    pulp.lpSum(
        df.iloc[i]["inventory_demand"] * x[i]
        for i in range(n)
    )
    <= inventory_capacity
)

In [10]:
max_delivery_time = 5

for i in range(n):

    model += (
        df.iloc[i]["delivery_time"] * x[i]
        <= max_delivery_time
    )

## Performing the Model

In [11]:
status = model.solve()

print("Status:", pulp.LpStatus[status])

Status: Optimal


## Selecting the No of Orders

In [12]:
selected_orders = []

for i in range(n):

    if x[i].value() == 1:

        selected_orders.append(i)

print(selected_orders)

[1512, 2366, 9764, 11196, 13063, 22922, 23319, 28972, 32650, 39689, 42050, 42392, 43952, 44352, 52213, 54467, 54702, 55320, 69297, 75083, 76954, 81100, 85993, 96489, 99771, 101485, 116098, 118839, 136881, 137151, 138195, 139275, 145858, 148580, 152209, 172747, 176636, 176712, 179758]


In [13]:
df.shape

(180519, 7)

## Objective Value 

In [14]:
print("Objective value =", pulp.value(model.objective))

Objective value = -1970.6400475204002


## Optimal Solutions for Objective Functions

In [15]:
selected_df = df.iloc[selected_orders]

print("Total Delivery Time:", selected_df["delivery_time"].sum())
print("Total Profit:", selected_df["profit"].sum())
print("Total Inventory Demand:", selected_df["inventory_demand"].sum())

Total Delivery Time: 62
Total Profit: 4988.600118801
Total Inventory Demand: 9999.350280110999


## Saving the output

In [16]:
milp_results = selected_df.copy()

milp_results.to_csv("MILP_Selected_Orders.csv", index=False)